In [1]:

const fs = await import("node:fs");

const years = Array.from(
  { length: 12 },
  (_, i) => String(2016 + i)
);

// Read the CSV file
const csv = fs.readFileSync(
  "UniversityRankings.csv",
  "utf8"
);

// Parse CSV, including quoted fields and commas
function parseCSV(text) {
  const rows = [];
  let row = [];
  let field = "";
  let quoted = false;

  text = text.replace(/^\uFEFF/, "");

  for (let i = 0; i < text.length; i++) {
    const c = text[i];

    if (c === '"') {
      if (quoted && text[i + 1] === '"') {
        field += '"';
        i++;
      } else {
        quoted = !quoted;
      }
    } else if (c === "," && !quoted) {
      row.push(field);
      field = "";
    } else if ((c === "\n" || c === "\r") && !quoted) {
      if (c === "\r" && text[i + 1] === "\n") i++;
      row.push(field);
      if (row.some(v => v.trim() !== "")) rows.push(row);
      row = [];
      field = "";
    } else {
      field += c;
    }
  }

  row.push(field);
  if (row.some(v => v.trim() !== "")) rows.push(row);

  const headers = rows.shift().map(h => h.trim());

  return rows.map(values =>
    Object.fromEntries(
      headers.map((h, i) => [h, values[i] ?? ""])
    )
  );
}

const data = parseCSV(csv);

// Convert rankings to numbers
const rankings = data.map(row => {
  const university = {
    name: row["University Name"],
    ipeds: row["IPEDS"]
  };

  years.forEach(year => {
    const value = String(row[year] ?? "").trim();
    const number = value === "" ? NaN : Number(value);

    university[year] = Number.isFinite(number)
      ? number
      : null;
  });

  return university;
});

// Select universities ranked 50 or better in 2016
const top50 = rankings
  .filter(row =>
    row["2016"] !== null &&
    row["2016"] <= 50
  )
  .sort((a, b) => a["2016"] - b["2016"]);

const universities = top50.map(row => row.name);

// Ranking values for blue cells
const z = top50.map(row =>
  years.map(year => {
    const value = row[year];
    return value !== null && value <= 50
      ? value
      : null;
  })
);

// All displayed labels
const labels = top50.map(row =>
  years.map(year => {
    const value = row[year];

    if (value === null) return "";
    if (value > 50) return ">50";

    return String(Math.trunc(value));
  })
);

// Gray background for every cell
const background = {
  z: top50.map(() => years.map(() => 1)),
  x: years,
  y: universities,
  type: "heatmap",
  colorscale: [
    [0, "lightgrey"],
    [1, "lightgrey"]
  ],
  showscale: false,
  hoverinfo: "skip",
  xgap: 1,
  ygap: 1
};

// Blue cells for ranks 1 through 50

const heatmap = {
  z: z.map(row => row.map(v => v === null ? null : 51 - v)),
  x: years,
  y: universities,
  type: "heatmap",

  colorscale: [
    [0.00, "#F7FBFF"],
    [0.20, "#DEEBF7"],
    [0.40, "#BDD7E7"],
    [0.60, "#6BAED6"],
    [0.80, "#2171B5"],
    [1.00, "#08306B"]
  ],

  zmin: 1,
  zmax: 50,

  colorbar: {
    title: "Rank",
    tickvals: [1, 11, 21, 31, 41, 50],
    ticktext: ["50", "40", "30", "20", "10", "1"]
  },

  xgap: 1,
  ygap: 1
};

// Display >50 on gray cells
const annotations = [];

top50.forEach((row, i) => {
  years.forEach(year => {
    const value = row[year];

    let label = "";
    let fontColor = "black";

    if (value === null) {
      label = "";
    } else if (value > 50) {
      label = ">50";
      fontColor = "black";
    } else {
      label = String(Math.trunc(value));

      // White text for the two darkest blue ranges
      if (value <= 20) {
        fontColor = "white";
      } else {
        fontColor = "black";
      }
    }

    if (label !== "") {
      annotations.push({
        x: year,
        y: universities[i],
        text: label,
        showarrow: false,
        font: {
          size: 10,
          color: fontColor
        }
      });
    }
  });
});

// Chart layout
const layout = {
  title: "University Rankings (2016–2027)",
  width: 1100,
  height: Math.max(800, universities.length * 28),
  margin: {
    l: 290,
    r: 100,
    t: 70,
    b: 80
  },
  
xaxis: {
  title: "Year",
  tickmode: "array",
  tickvals: years,
  ticktext: years,
  tickangle: -45,
  side: "bottom"
},
  yaxis: {
    title: "University",
    autorange: "reversed",
    type: "category",
    categoryorder: "array",
    categoryarray: universities
  },
  annotations: annotations,
  plot_bgcolor: "lightgrey"
};

// Generate an HTML visualization from JavaScript
const html = `
<!DOCTYPE html>
<html>
<head>
<meta charset="UTF-8">
<title>University Rankings Heatmap</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
</head>
<body>
<div id="heatmap"></div>
<script>
const background = ${JSON.stringify(background)};
const heatmap = ${JSON.stringify(heatmap)};
const layout = ${JSON.stringify(layout)};

Plotly.newPlot(
  "heatmap",
  [background, heatmap],
  layout
);
</script>
</body>
</html>
`;

fs.writeFileSync(
  "UniversityRankingsHeatmap.html",
  html
);

console.log("Heatmap created successfully!");
console.log("Universities included:", top50.length);
console.log("Open UniversityRankingsHeatmap.html in Chrome.");



Heatmap created successfully!
Universities included: 50
Open UniversityRankingsHeatmap.html in Chrome.


In [7]:


const { createRequire } = await import("node:module");
const require = createRequire(process.cwd() + "/package.json");

const fs = require("node:fs");
const { parse } = require("csv-parse/sync");

// ======================================
// 1. READ CSV FILES
// ======================================

let UniversityRankings = parse(
    fs.readFileSync("UniversityRankings.csv", "utf8"),
    { columns: true, skip_empty_lines: true, bom: true }
);

const ipeds = parse(
    fs.readFileSync("hd2024.csv", "utf8"),
    { columns: true, skip_empty_lines: true, bom: true }
);


// ======================================
// 2. CLASSIFY PUBLIC AND PRIVATE SCHOOLS
// ======================================

const typeMap = {
    1: "Public",
    2: "Private Nonprofit",
    3: "Private For-Profit"
};

const ipedsMap = new Map();

for (const row of ipeds) {
    const id = String(row.UNITID).trim();

    if (ipedsMap.has(id)) {
        throw new Error(`Duplicate IPEDS UNITID: ${id}`);
    }

    ipedsMap.set(
        id,
        typeMap[Number(row.CONTROL)] || null
    );
}

UniversityRankings = UniversityRankings.map(row => ({
    ...row,
    UNITID: row.IPEDS,
    Type: ipedsMap.get(String(row.IPEDS).trim()) || null
}));

// ======================================
// 3. FILTER PUBLIC UNIVERSITIES
// ======================================

let Public = UniversityRankings.filter(
    row => row.Type === "Public"
);

console.log("Total public universities:", Public.length);

// ======================================
// 4. CONVERT RANKINGS TO NUMBERS
// ======================================

const years = [
    "2016", "2017", "2018", "2019",
    "2020", "2021", "2022", "2023",
    "2024", "2025", "2026", "2027"
];

Public = Public.map(row => {
    const result = { ...row };

    for (const year of years) {
        const value = String(row[year] ?? "").trim();

        result[year] =
            value !== "" && Number.isFinite(Number(value))
                ? Number(value)
                : null;
    }

    return result;
});

const texasAM = Public.find(row =>
    row["University Name"].includes("Texas A&M University-College Station")
);

console.log("Texas A&M national rank in 2023:", texasAM["2023"]);

const higherRanked = Public.filter(row =>
    row["2023"] !== null &&
    row["2023"] < texasAM["2023"]
);

console.log("Public universities ranked higher:", higherRanked.length);

// data not as number as expected, so we will manually set Texas A&M's 2023 rank to 26

Public = Public.map(row => {
    if (
        row["University Name"] ===
        "Texas A&M University-College Station"
    ) {
        return {
            ...row,
            "2023": 67
        };
    }

    return row;
});

// ======================================
// 5. RANK PUBLIC UNIVERSITIES EACH YEAR
// ======================================

for (const year of years) {

    const sortedRanks = Public
        .map(row => row[year])
        .filter(value => value !== null)
        .sort((a, b) => a - b);

    const rankingMap = new Map();

    sortedRanks.forEach((value, index) => {
        if (!rankingMap.has(value)) {
            rankingMap.set(value, index + 1);
        }
    });

    Public = Public.map(row => ({
        ...row,
        [year]: row[year] === null
            ? null
            : rankingMap.get(row[year])
    }));
}

// ======================================
// 6. SELECT TOP 50 PUBLIC UNIVERSITIES
// BASED ON 2016
// ======================================

const Top50Public = Public
    .filter(row => row["2016"] !== null && row["2016"] <= 50)
    .sort((a, b) => a["2016"] - b["2016"]);

console.log(
    "Number of top 50 public universities:",
    Top50Public.length
);

// ======================================
// 7. PREPARE HEATMAP DATA
// ======================================

const universityNames = Top50Public.map(
    row => row["University Name"]
);

// Reverse the numeric values for the color bar:
// Rank 1 = 50
// Rank 50 = 1
// Rank >50 or missing = 0

const z = Top50Public.map(row =>
    years.map(year => {
        const rank = row[year];

        return rank === null || rank > 50
            ? 0
            : 51 - rank;
    })
);

// Keep actual rankings as displayed labels

const labels = Top50Public.map(row =>
    years.map(year => {
        const rank = row[year];

        return rank === null || rank > 50
            ? ">50"
            : String(Math.trunc(rank));
    })
);

// ======================================
// 8. CREATE HEATMAP
// ======================================

// Explicitly define the reversed color scale.
// Bottom = grey / light blue
// Top = dark blue

const blueScale = [
    [0.00, "#d3d3d3"],
    [0.0199, "#d3d3d3"],
    [0.0200, "#f7fbff"],
    [0.20, "#c6dbef"],
    [0.40, "#9ecae1"],
    [0.60, "#6baed6"],
    [0.80, "#2171b5"],
    [1.00, "#08306b"]
];

const heatmap = {
    type: "heatmap",

    x: years,
    y: universityNames,
    z: z,

    text: labels,

    colorscale: blueScale,

    zmin: 0,
    zmax: 50,

    xgap: 1,
    ygap: 1,

    hovertemplate:
        "University: %{y}<br>" +
        "Year: %{x}<br>" +
        "Public Rank: %{text}<extra></extra>",

    colorbar: {
        title: {
            text: "Public Rank"
        },

        tickmode: "array",

        // Reversed values with actual ranking labels
        tickvals: [50, 41, 31, 21, 11, 1],
        ticktext: ["1", "10", "20", "30", "40", "50"],

        thickness: 20,
        len: 0.85
    },

    showscale: true
};

// ======================================
// 9. CREATE TEXT LABELS
// ======================================

// Use scatter text traces rather than annotations.
// This gives us independent text colors without
// interfering with the heatmap cells.

const whiteX = [];
const whiteY = [];
const whiteText = [];

const blackX = [];
const blackY = [];
const blackText = [];

Top50Public.forEach((row, i) => {
    years.forEach((year, j) => {

        const rank = row[year];
        const label = labels[i][j];

        // First two darkest blue shades
        if (rank !== null && rank <= 21) {
            whiteX.push(year);
            whiteY.push(universityNames[i]);
            whiteText.push(label);
        } else {
            blackX.push(year);
            blackY.push(universityNames[i]);
            blackText.push(label);
        }
    });
});

const whiteLabels = {
    type: "scatter",
    mode: "text",

    x: whiteX,
    y: whiteY,
    text: whiteText,

    textfont: {
        color: "white",
        size: 10
    },

    hoverinfo: "skip",
    showlegend: false
};

const blackLabels = {
    type: "scatter",
    mode: "text",

    x: blackX,
    y: blackY,
    text: blackText,

    textfont: {
        color: "black",
        size: 10
    },

    hoverinfo: "skip",
    showlegend: false
};

// ======================================
// 10. HEATMAP LAYOUT
// ======================================

const layout = {
    title: {
        text: "Top 50 public universities (2016–2027)",
        font: {
            size: 16
        }
    },

    xaxis: {
        title: "Year",
        tickangle: -45,
        side: "bottom",
        type: "category",

        categoryorder: "array",
        categoryarray: years,

        showgrid: false
    },

    yaxis: {
        title: "Public university",

        autorange: "reversed",
        type: "category",

        categoryorder: "array",
        categoryarray: universityNames,

        automargin: true,
        showgrid: false
    },

    width: 1100,

    height: Math.max(
        1200,
        Top50Public.length * 27
    ),

    margin: {
        l: 310,
        r: 120,
        t: 80,
        b: 100
    },

    paper_bgcolor: "white",
    plot_bgcolor: "white",

    showlegend: false
};

// ======================================
// 11. SAVE INTERACTIVE HEATMAP AS HTML
// ======================================

const plotlyPath = require.resolve(
    "plotly.js-dist-min"
);

const plotlyCode = fs.readFileSync(
    plotlyPath,
    "utf8"
);

const html = `
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>Public University Rankings Heatmap</title>
</head>
<body>

<div id="heatmap" style="width:100%;"></div>

<script>${plotlyCode}</script>

<script>
    const data = ${JSON.stringify([
        heatmap,
        whiteLabels,
        blackLabels
    ])};

    const layout = ${JSON.stringify(layout)};

    Plotly.newPlot(
        "heatmap",
        data,
        layout,
        {
            responsive: true,
            displayModeBar: true,
            toImageButtonOptions: {
                format: "png",
                filename: "public_university_heatmap",
                scale: 2
            }
        }
    );
</script>

</body>
</html>
`;

fs.writeFileSync(
    "public_university_heatmap.html",
    html
);

console.log(
    "Heatmap saved as public_university_heatmap.html"
);


Total public universities: 101
Texas A&M national rank in 2023: null
Public universities ranked higher: 0
Number of top 50 public universities: 51
Heatmap saved as public_university_heatmap.html


In [10]:

const { createRequire } = await import("node:module");
const require = createRequire(process.cwd() + "/package.json");

const fs = require("node:fs");
const { parse } = require("csv-parse/sync");

// ======================================
// 1. READ CSV FILE
// ======================================

let UniversityRankings = parse(
    fs.readFileSync("UniversityRankings.csv", "utf8"),
    {
        columns: true,
        skip_empty_lines: true,
        bom: true
    }
);

// ======================================
// 2. SELECT ALL UC UNIVERSITIES
// ======================================

const years = [
    "2016", "2017", "2018",
    "2019", "2020", "2021", "2022", "2023", "2024",
    "2025", "2026", "2027"
];

// Include every UC campus found in the CSV
let UC_data = UniversityRankings.filter(row =>
    String(row["University Name"] ?? "")
        .toLowerCase()
        .includes("university of california")
);

// ======================================
// 3. CONVERT RANKINGS TO NUMBERS
// ======================================

UC_data = UC_data.map(row => {
    const result = { ...row };

    for (const year of years) {
        const value = String(row[year] ?? "").trim();

        result[year] =
            value !== "" && Number.isFinite(Number(value))
                ? Number(value)
                : null;

        if (value !== "" && result[year] === null) {
            console.warn(
                `Invalid ranking: ${row["University Name"]}, ${year}: ${value}`
            );
        }
    }

    return result;
});

// ======================================
// 4. SORT BY 2022 RANKING
// ======================================

UC_data.sort((a, b) =>
    (a["2016"] ?? Infinity) -
    (b["2016"] ?? Infinity)
);

console.log(
    "Number of UC universities:",
    UC_data.length
);

console.log(
    "UC campuses:",
    UC_data.map(row => row["University Name"])
);

// ======================================
// 5. PREPARE HEATMAP DATA
// ======================================

const universityNames = UC_data.map(
    row => row["University Name"]
);

// Reverse values to put rank 1 at the
// top of the color bar.
// Missing values remain null.

const allRanks = UC_data.flatMap(row =>
    years.map(year => row[year])
).filter(value => value !== null);

if (allRanks.length === 0) {
    throw new Error("No valid UC rankings found.");
}

const maxRank = Math.max(...allRanks);

// Higher values = better rankings
const z = UC_data.map(row =>
    years.map(year =>
        row[year] === null
            ? null
            : maxRank + 1 - row[year]
    )
);

// Actual numbers displayed inside cells
const labels = UC_data.map(row =>
    years.map(year =>
        row[year] === null
            ? "N/A"
            : String(Math.trunc(row[year]))
    )
);

// ======================================
// 6. CREATE BLUE COLOR SCALE
// ======================================

// Low transformed value = light blue
// High transformed value = dark blue

const blueScale = [
    [0.00, "#f7fbff"],
    [0.20, "#c6dbef"],
    [0.40, "#9ecae1"],
    [0.60, "#6baed6"],
    [0.80, "#2171b5"],
    [1.00, "#08306b"]
];

// ======================================
// 7. CREATE HEATMAP
// ======================================

const heatmap = {
    type: "heatmap",

    x: years,
    y: universityNames,
    z: z,

    text: labels,

    colorscale: blueScale,
    zmin: 1,
    zmax: maxRank,

    xgap: 1,
    ygap: 1,

    hovertemplate:
        "University: %{y}<br>" +
        "Year: %{x}<br>" +
        "National Rank: %{text}<extra></extra>",

    colorbar: {
        title: {
            text: "Ranking"
        },

        tickmode: "array",
        tickvals: [maxRank, maxRank * 0.75, maxRank * 0.5, maxRank * 0.25, 1],
        ticktext: [
            "1",
            String(Math.round(maxRank * 0.25 + 1)),
            String(Math.round(maxRank * 0.5 + 1)),
            String(Math.round(maxRank * 0.75 + 1)),
            String(maxRank)
        ],

        thickness: 20,
        len: 0.9
    },

    showscale: true
};

// ======================================
// 8. CREATE WHITE AND BLACK TEXT LABELS
// ======================================

const whiteX = [];
const whiteY = [];
const whiteText = [];

const blackX = [];
const blackY = [];
const blackText = [];

UC_data.forEach((row, i) => {
    years.forEach((year, j) => {
        const rank = row[year];
        const label = labels[i][j];

        if (rank === null) {
            blackX.push(year);
            blackY.push(universityNames[i]);
            blackText.push("N/A");
            return;
        }

        // White text on two darkest shades
        const normalized =
            maxRank > 1
                ? (maxRank - rank) / (maxRank - 1)
                : 1;

        if (normalized >= 0.60) {
            whiteX.push(year);
            whiteY.push(universityNames[i]);
            whiteText.push(label);
        } else {
            blackX.push(year);
            blackY.push(universityNames[i]);
            blackText.push(label);
        }
    });
});

const whiteLabels = {
    type: "scatter",
    mode: "text",

    x: whiteX,
    y: whiteY,
    text: whiteText,

    textfont: {
        color: "white",
        size: 12
    },

    hoverinfo: "skip",
    showlegend: false
};

const blackLabels = {
    type: "scatter",
    mode: "text",

    x: blackX,
    y: blackY,
    text: blackText,

    textfont: {
        color: "black",
        size: 12
    },

    hoverinfo: "skip",
    showlegend: false
};

// ======================================
// 9. HEATMAP LAYOUT
// ======================================

const layout = {
    title: {
        text: "UC rankings comparison (2016–2027)",
        font: {
            size: 18
        }
    },

    xaxis: {
        title: "Year",
        tickangle: -45,
        type: "category",
        categoryorder: "array",
        categoryarray: years,
        showgrid: false
    },

    yaxis: {
        title: "University",
        autorange: "reversed",
        type: "category",
        categoryorder: "array",
        categoryarray: universityNames,
        automargin: true,
        showgrid: false
    },

    width: 1100,
    height: Math.max(500, UC_data.length * 48),

    margin: {
        l: 310,
        r: 120,
        t: 80,
        b: 100
    },

    paper_bgcolor: "white",
    plot_bgcolor: "white",

    showlegend: false
};

// ======================================
// 10. SAVE INTERACTIVE HEATMAP AS HTML
// ======================================

const plotlyPath = require.resolve(
    "plotly.js-dist-min"
);

const plotlyCode = fs.readFileSync(
    plotlyPath,
    "utf8"
);

const html = `
<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <title>UC Rankings Heatmap</title>
</head>
<body>

<div id="heatmap" style="width:100%;"></div>

<script>${plotlyCode}</script>

<script>
    const data = ${JSON.stringify([
        heatmap,
        whiteLabels,
        blackLabels
    ])};

    const layout = ${JSON.stringify(layout)};

    Plotly.newPlot(
        "heatmap",
        data,
        layout,
        {
            responsive: true,
            displayModeBar: true,
            toImageButtonOptions: {
                format: "png",
                filename: "uc_rankings_heatmap",
                scale: 2
            }
        }
    );
</script>

</body>
</html>
`;

fs.writeFileSync(
    "uc_rankings_heatmap.html",
    html
);

console.log(
    "Heatmap saved as uc_rankings_heatmap.html"
);


Number of UC universities: 9


UC campuses: [
  'University of California-Berkeley',
  'University of California-Los Angeles',
  'University of California-Santa Barbara',
  'University of California-San Diego',
  'University of California-Irvine',
  'University of California-Davis',
  'University of California-Santa Cruz',
  'University of California-Riverside',
  'University of California-Merced'
]
Heatmap saved as uc_rankings_heatmap.html
